# Notebook 13 — Product geometry and spherical semantic routing in JEPA

## Purpose

Implement and compare the proposed geometry changes in the full custom JEPA training loop. The online encoder and EMA target encoder change together; masking, predictor, JEPA objective and matched training exposure stay fixed.

Six conditions: ViT; the original learned Gaussian kernel; per-scale normalization only; independent spatial/content length scales; spherical semantic coordinates with independent spatial scale; and three small semantic coordinate views with soft memberships.

These are prescribed geometric embeddings for patch interactions. They do not establish manifold charts or an isomorphism between images and representations. Kernels are computed densely; FMM acceleration is not claimed.

## Inputs and outputs

Inputs: audited Food-101 data and the sealed original 10,100 training / 2,020 development membership. The previously opened 8,074-image reserve is development data and is not described as fresh. The official 25,246-image test lockbox remains separate and unused.

Outputs: registered source and configuration hashes, complete paired training checkpoints, fixed-epoch linear probes, Holm-corrected paired-seed comparisons, geometry diagnostics, compute/memory measurements and a saved visual report.

The default design has eight fresh paired seeds, six variants and initialization/final probes. The common final update count is chosen from registered candidates using discarded timing and memory measurements only, before quality outcomes. Shorter runtime configurations are declared separately before any run starts.

## Runtime, checkpoints and resume

Expected runtime: timing-calibrated within the selected budget, default **up to 12 hours** including preflight, training, probing and reporting. Run All starts or reconnects to the actual detached study. The original clock never resets.

Checkpoints: `checkpoints/geometry-ablation-001/`; latest two valid full-state checkpoints and best are retained for every arm. Saves occur at stage/epoch boundaries and at most every 50 updates or 15 minutes. Model, EMA target, optimizer/scheduler, every RNG, consumed sample/mask position, configuration and metrics are saved atomically. One writer per run; invalid manifests raise an error.

Exact resume: open this notebook in a fresh `ahr-go` kernel and Run All with the same run ID and unchanged sources before the original deadline. The launcher validates the committed state and resumes the saved update/cursor. It never substitutes a new run for invalid state. Exhausted deadlines and missing endpoints remain incomplete/UNKNOWN.

Detached session: `fmm-product-geometry-geometry-ablation-001`; attach with `tmux attach -t fmm-product-geometry-geometry-ablation-001`, detach with Ctrl-b then d. Hourly agent checks and a separate deadline watcher supervise operational health.

Storage is the persistent boot disk. The instance-deletion policy is unverified; copy and hash-verify final checkpoints off-instance before deleting or recycling it.


## Statistical comparisons fixed before training

The primary family contains three final comparisons against ViT: independent spatial/content scales, spherical semantic coordinates, and multiple coordinate views. Use the paired training seed as the independent replication unit, two-sided paired t tests, and Holm correction at family alpha 0.05. Raw 95% intervals and conservative simultaneous intervals are labelled separately; sign flips and sign tests are robustness checks under their stated assumptions.

The old kernel and normalization-only condition are mechanism controls. Initialization gains, geometry/rank diagnostics and costs are secondary. No interim quality-based stopping, best-seed selection or repeated significance search is allowed. Small p values are possible, not guaranteed. Development reuse limits claims; this is not an official-test confirmation.


## Kernel encoder mathematics

Every head keeps two fixed spatial coordinates and six learned semantic coordinates. The original learned kernel normalizes after mixing three Gaussian scales. The normalization-only control instead normalizes each scale before using the same learned scale coefficients.

The product variant uses independent global positive spatial and semantic lengths for each head and scale:

$$K_s(i,j)=\exp\left[-\frac{\|p_i-p_j\|^2}{2\ell_{p,s}^2}-\frac{\|c_i^q-c_j^k\|^2}{2\ell_{c,s}^2}\right].$$

The spherical variant replaces bounded semantic coordinates with unit six-vectors on $S^5$, using the chordal kernel $\exp[-(1-u_i^q\cdot u_j^k)/\ell_{c,s}^2]$ together with the spatial factor. Exactly zero semantic vectors use a declared deterministic unit-vector fallback with a guarded derivative.

The multiple-view variant uses three small semantic coordinate maps. Positive source memberships enter each view's normalization; receiver memberships mix the view outputs. In each branch,

$$P_{ij}^{a,s}=\frac{K_{a,s}(i,j)\rho_a(h_j)}{\sum_{k\in C}K_{a,s}(i,k)\rho_a(h_k)},\qquad m_i=\sum_a g_a(h_i)\sum_s\pi_s\sum_{j\in C}P_{ij}^{a,s}v_j.$$

Single-view conditions have $g=\rho=1$. Only valid visible patches enter a context encoder. Both online and EMA target encoders change together; output features remain 192-dimensional and the predictor, objective, target normalization, residuals and pointwise MLPs stay fixed.

Query and source maps remain separate, so routing can be directed. These embeddings do not establish a recovered symmetric intrinsic token metric or verified manifold atlas. Dense kernels isolate representation quality; FMM approximation is separate work.

Shared execution optimizations and ordered preprocessing preserve the original computation and apply to every condition. They are not evidence of a kernel-model speed advantage.

## Operational recovery and visual review

No labels enter the self-supervised JEPA loss. Fixed-epoch linear classifiers are trained only after extracting frozen EMA features, with feature standardization fitted on probe-training images alone.

The report is saved at artifacts/notebook_13/geometry-ablation-001/stages/learning/visual_report/report.html; open notebooks/13_Results_Review.ipynb for a read-only view. It shows accuracy/gains, corrected primary comparisons, full-step timing and memory, learned distance scales, mixing diagnostics and actual development image examples.

Before long training, four uninterrupted updates versus an interrupted two-plus-two continuation must be bitwise equal for every condition. Real feature/probe integration must also pass. Completed training evidence is reused during evaluation recovery.

The worker is fmm-product-geometry-geometry-ablation-001. Attach with tmux attach -t fmm-product-geometry-geometry-ablation-001; detach with Ctrl-b then d. The independent deadline and heartbeat sessions use the same run ID. GPU 0 is the configured device.

Hourly reviews can create a separately bound operational recovery adapter after validation, but cannot change models, data, seeds, optimization, quality endpoints, registered source hashes or the original clock. Logs live in heartbeat/agent_reviews/; current health is in heartbeat/latest_heartbeat.json.


## 1. Load tagged definitions

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import nbformat, importlib.util, sys
from IPython.display import display, Markdown, Image
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm13_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/13_product_geometry/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 2. Registered configuration
Eight fresh paired seeds, six conditions, timing-only update candidates, initialization/final probes and the 12-hour cap are fixed before outcomes. Three geometry-versus-ViT comparisons form the primary Holm-corrected family.

In [ ]:
RUN_ID = "geometry-ablation-001"
MAX_SECONDS = 12 * 60 * 60
CONFIG = WORKFLOW.study_configuration(RUN_ID, budget_seconds=MAX_SECONDS)
CONFIG


## 3. Readiness
Read-only check of operator, input, execution, pipeline, visual, supervisor and live-agent certificates. It creates no experiment clock or worker.

In [ ]:
READINESS = WORKFLOW.readiness(ROOT, CONFIG)
import pandas as pd
display(pd.DataFrame(READINESS["architecture_summary"]))
display({key: READINESS[key] for key in ("status", "production_run_started", "expected_runtime", "planned_training_runs", "planned_probe_evaluations", "checkpoint_location", "scientific_results")})


## 4. Launch or reconnect
This cell starts the real overnight study. All preflights, training, evaluations and visual report generation run independently in tmux under the original cap.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
DISPLAY_LAUNCH = {key: value for key, value in LAUNCH.items() if key != "command"}
if LAUNCH.get("deadline_unix"):
    DISPLAY_LAUNCH["hard_stop_dublin"] = datetime.fromtimestamp(LAUNCH["deadline_unix"], ZoneInfo("Europe/Dublin")).isoformat()
display(DISPLAY_LAUNCH)


## 5. Inspect progress and heartbeat
Rerun this cell for checkpoint counts and the latest server-side heartbeat.

In [ ]:
STATUS = WORKFLOW.inspect_study(ROOT, RUN_ID)
PROGRESS = STATUS.get("saved_training_progress", [])
display({key: value for key, value in STATUS.items() if key != "saved_training_progress"})
if PROGRESS:
    display(pd.DataFrame(PROGRESS))

HEARTBEAT=ROOT/'artifacts/notebook_13'/RUN_ID/'heartbeat/latest_heartbeat.json'
if HEARTBEAT.exists():
    import json
    LIVE_HEARTBEAT=json.loads(HEARTBEAT.read_text())
    display({k:LIVE_HEARTBEAT.get(k) for k in ('observed_unix','worker_alive','activity_age_seconds','remaining_seconds','gpu_csv','evaluation_progress')})
PLAN_PATH=ROOT/'artifacts/notebook_13'/RUN_ID/'stages/learning/plan.json'
if PLAN_PATH.exists():
    PLAN=json.loads(PLAN_PATH.read_text())
    display({'selected_final_updates':PLAN['updates'],'milestones':PLAN['milestones'],'budget_selection':PLAN['budget_selection']})


## 6. Read completed results
The separate read-only results notebook and embedded HTML report give the complete visual review.

In [ ]:

REPORT_FOLDER=ROOT/'artifacts/notebook_13'/RUN_ID/'stages/learning'
REPORT=REPORT_FOLDER/'stage_results.md'
if STATUS.get('status')=='COMPLETED' and REPORT.exists():
    display(Markdown('\n'.join(line for line in REPORT.read_text().splitlines() if not line.startswith('!['))))
    display(Markdown('Open **notebooks/13_Results_Review.ipynb** for the image examples and full visual report.'))
    for figure in ('stage_quality.png','stage_compute.png','stage_contrasts.png'):
        p=REPORT_FOLDER/figure
        if p.exists():display(Image(filename=str(p)))
else:
    display(Markdown('**Study is still running or incomplete.** Open the read-only results notebook to inspect progress. Missing comparisons remain unknown; partial scores do not establish a winner.'))


## Interpretation

Assess the three geometry-versus-ViT comparisons using Holm-adjusted p values and paired effect directions across all eight training seeds. Raw 95% and conservative simultaneous 98.333% intervals are labelled separately. Constant-difference cases with undefined variance inference remain unsupported.

Initialization gains and the original/normalization-only controls help interpret mechanisms. Measured training-step latency and allocated GPU memory describe this implementation; parameter count alone establishes no speed advantage.

The images are reused development data. A positive result would support a bounded architecture comparison under this custom JEPA protocol; an official-test confirmation remains separate. Missing endpoints and deadline cutoffs stay UNKNOWN. No FMM speedup, manifold recovery or equivalence follows from these plots.

References: [I-JEPA](https://arxiv.org/abs/2301.08243), [Fast Multipole Attention](https://arxiv.org/abs/2310.11960).